# Merge Embryos

## Purpose

After downloading per embryo h5td files from [zenodo](https://doi.org/10.5281/zenodo.19892784), this notebook merges them into combined h5td files used for subsequent analysis.

## Setup

In [ ]:
import pandas as pd
import scanpy as sc
import treedata as td
from devmap.config import embryos, get_paths

base_path, plots_path, results_path = get_paths("data")
data_path = base_path / "data"

## Load data

In [ ]:
tdatas = []
for embryo in embryos:
    tdatas.append(td.read_h5td(data_path / "embryos" /f"{embryo}.h5td"))
tdata = td.concat(tdatas)
del tdatas

## Save counts

In [ ]:
tdata.write_h5td(data_path / "counts.h5td")

## Save normalized counts

In [ ]:
sc.pp.normalize_total(tdata, target_sum=2e4)
sc.pp.log1p(tdata)
tdata.write_h5td(data_path / "log1p.h5td")

Subet to highly variable genes

In [ ]:
hvg = pd.read_csv(data_path / "hvg.csv", index_col=0, header=None).index
tdata[:,hvg].write_h5td(data_path / "log1p_hvg.h5td")

## Save topology

In [ ]:
tdata[[],:].write_h5td(data_path / "topology.h5td")

## Save CSV files

In [ ]:
tdata.obsm["X_scvi"].write_csv(data_path / "scvi.csv")
tdata.obsm["X_umap"].write_csv(data_path / "umap.csv")
tdata.obs.write_csv(data_path / "obs.csv")